# Fase 1: Pipeline de Curación de Datos Médicos (ENSANUT)
**Objetivo:** Limpiar anomalías gubernamentales, rescatar datos de segundas mediciones e imputar valores faltantes mediante algoritmos de vecinos más cercanos (KNN) para preservar la integridad estadística antes del modelado predictivo.

In [ ]:
import pandas as pd
import numpy as np
import kagglehub
import os
from sklearn.impute import KNNImputer

# Descargar dataset de kagglehub
path = kagglehub.dataset_download("frederickfelix/hipertensin-arterial-mxico")

csv_file = os.path.join(path, 'Hipertension_Arterial_Mexico.csv')

df = pd.read_csv(csv_file)

print(f"Dimensión inicial: {df.shape[0]} pacientes y {df.shape[1]} variables.")
df.head(3) # Mostrará una tabla estilizada con los primeros 3 pacientes

Dimensión inicial: 4363 pacientes y 36 variables.


,FOLIO_I,sexo,edad,concentracion_hemoglobina,temperatura_ambiente,valor_acido_urico,valor_albumina,valor_colesterol_hdl,valor_colesterol_ldl,valor_colesterol_total,...,segundamedicion_peso,segundamedicion_estatura,distancia_rodilla_talon,circunferencia_de_la_pantorrilla,segundamedicion_cintura,tension_arterial,sueno_horas,masa_corporal,actividad_total,riesgo_hipertension
0,2022_01001004,2,41,14.2,22,4.8,4.0,34,86.0,139,...,64.70,154.0,48.5,33.5,0.0,107,4,32.889389,120,1
1,2022_01001009,2,65,14.1,9,4.4,3.8,73,130.0,252,...,96.75,152.2,44.5,41.1,113.7,104,2,1.000000,240,0
2,2022_01001012,2,68,14.2,22,4.8,4.0,34,86.0,139,...,68.70,144.8,42.3,37.8,103.7,105,1,1.000000,480,0


### Fase A: Tratamiento de Anomalías
Identificamos "trampas de datos": adultos con pesos irreales (<30kg) o medidas de cintura de 0cm. Convertiremos estos errores en valores nulos reales (`NaN`) para no sesgar el algoritmo.

In [4]:
# Convertir imposibilidades físicas a NaN
df.loc[df['peso'] < 30, 'peso'] = np.nan
df.loc[df['medida_cintura'] == 0, 'medida_cintura'] = np.nan
df.loc[df['masa_corporal'] < 10, 'masa_corporal'] = np.nan 

print(f"Total de pesos marcados como nulos: {df['peso'].isna().sum()}")

Total de pesos marcados como nulos: 965


### Fase B: Rescate mediante Columnas Auxiliares
Si el peso principal es nulo pero existe una 'segunda medicion' válida, rescataremos ese dato en lugar de perder al paciente completo.

In [5]:
# Rescate de peso
condicion_rescate_peso = df['peso'].isna() & (df['segundamedicion_peso'] >= 30)
df.loc[condicion_rescate_peso, 'peso'] = df.loc[condicion_rescate_peso, 'segundamedicion_peso']

# Rescate de cintura
condicion_rescate_cintura = df['medida_cintura'].isna() & (df['segundamedicion_cintura'] > 0)
df.loc[condicion_rescate_cintura, 'medida_cintura'] = df.loc[condicion_rescate_cintura, 'segundamedicion_cintura']

print(f"Pesos nulos restantes después del rescate: {df['peso'].isna().sum()}")

Pesos nulos restantes después del rescate: 6


### Fase C: Imputación Inteligente (Machine Learning)
Para los valores que aún son nulos, usaremos `KNNImputer`. En lugar de un promedio global, buscaremos los 5 pacientes "gemelos estadísticos" (por edad, sexo, estatura) y promediaremos sus valores para rellenar los huecos.

In [6]:
# Definir variables para buscar "gemelos"
columnas_para_imputar = ['sexo', 'edad', 'estatura', 'peso', 'medida_cintura']

# Configurar e instanciar el modelo KNN
imputer = KNNImputer(n_neighbors=5)
df_imputed = df.copy()

# Aplicar la imputación
df_imputed[columnas_para_imputar] = imputer.fit_transform(df[columnas_para_imputar])

# Actualizar el dataframe principal
df['peso'] = df_imputed['peso']
df['medida_cintura'] = df_imputed['medida_cintura']

print("Imputación completada. Ya no hay valores nulos en peso ni cintura.")

Imputación completada. Ya no hay valores nulos en peso ni cintura.


In [7]:
# Exportar la matriz limpia a la sala de procesados
output_path = "../data/processed/Hipertension_Limpio.csv"
df.to_csv(output_path, index=False)
print(f"Pipeline exitoso. Matriz guardada en: {output_path}")

Pipeline exitoso. Matriz guardada en: ../data/processed/Hipertension_Limpio.csv
